In [ ]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"             # never contact the Hugging Face Hub

import torch
from transformers import Sam3Model, Sam3Processor, BitsAndBytesConfig
from huggingface_hub.utils import GatedRepoError

# Load processor and model offline
processor = Sam3Processor.from_pretrained("models/sam3-4bit", local_files_only=True)
model = Sam3Model.from_pretrained("models/sam3-4bit", device_map="auto", local_files_only=True)

In [ ]:
import torch, numpy as np, torch.nn.functional as Fnn
import matplotlib.pyplot as plt, matplotlib.patches as patches
from PIL import Image

# ---------------- Settings ----------------
image_path  = "images/zil_00-10-83-13-99_261012_112259_Z.png"
text_prompt = "circurt module"
example_box = None                      # e.g. [x1, y1, x2, y2] in full-image pixels -> positive example
THR         = 0.5                       # score threshold
REL_THR     = None                      # e.g. 0.5 -> keep scores >= 0.5 * best score (overrides THR)
MIN_ONE     = True                      # if nothing passes, keep the single best candidate
DISPLAY_DIV = 4                         # 1 = full resolution masks; 4 = 1/4 size (faster, less RAM)
ALPHA       = 0.6

# ---------------- 1) Load height map and scale to 0..255 ----------------
raw = Image.open(image_path)
print("file mode:", raw.mode)           # I;16 = 16-bit heights, L = already 8-bit
z = np.array(raw).astype(np.float32)
if z.ndim == 3:                         # already RGB: use one channel
    z = z[..., 0]
valid = z > 0                           # 0 = no data (laser shadow)
lo, hi = np.percentile(z[valid], [1, 99.5])
g = np.clip((z - lo) / (hi - lo), 0, 1)
g[~valid] = 0
image = Image.fromarray((g * 255).astype(np.uint8)).convert("RGB")
W, H = image.size

# ---------------- 2) Run SAM 3 ----------------
proc_kwargs = dict(images=image, text=text_prompt, return_tensors="pt")
if example_box is not None:
    proc_kwargs.update(input_boxes=[[example_box]], input_boxes_labels=[[1]])
inputs = processor(**proc_kwargs).to(model.device)

with torch.inference_mode():
    outputs = model(**inputs)

# Scores = object score x presence score (same formula as the processor)
scores = outputs.pred_logits[0].float().sigmoid()
presence = None
if outputs.presence_logits is not None:
    presence = float(outputs.presence_logits[0].float().sigmoid())
    scores = scores * presence
scores = scores.cpu()

order = scores.argsort(descending=True)
print("presence score:", None if presence is None else round(presence, 3))
print("top 10 scores:", np.round(scores[order[:10]].numpy(), 3))

# ---------------- 3) Choose which instances to keep ----------------
if REL_THR is not None:
    keep = torch.nonzero(scores >= REL_THR * scores.max()).squeeze(1)
else:
    keep = torch.nonzero(scores > THR).squeeze(1)
forced = False
if keep.numel() == 0 and MIN_ONE:
    keep = order[:1]                    # fall back to the best candidate
    forced = True
keep = keep[scores[keep].argsort(descending=True)]   # highest score first
print(f"kept {len(keep)} instance(s)" + ("  (forced top-1, check it!)" if forced else ""))

# Copy what we need to the CPU, then free the GPU
mask_logits = outputs.pred_masks[0, keep].float().cpu()          # (K, h, w), low resolution
boxes_norm  = outputs.pred_boxes[0, keep].float().cpu()          # x1, y1, x2, y2, normalized 0..1
kept_scores = scores[keep]
best_box_px = (outputs.pred_boxes[0, order[0]].float().cpu() * torch.tensor([W, H, W, H])).tolist()
print("best box (px):", [round(v) for v in best_box_px])

del outputs, inputs
torch.cuda.empty_cache()

# ---------------- 4) Scale masks up on the CPU, one at a time ----------------
h, w = H // DISPLAY_DIV, W // DISPLAY_DIV
masks = np.zeros((len(keep), h, w), dtype=bool)
for i in range(len(keep)):
    m = Fnn.interpolate(mask_logits[i][None, None], size=(h, w), mode="bilinear", align_corners=False)
    masks[i] = (m[0, 0] > 0).numpy()                             # logit > 0 == probability > 0.5
masks &= valid[::DISPLAY_DIV, ::DISPLAY_DIV][:h, :w]             # drop no-data pixels from masks
boxes_px = (boxes_norm * torch.tensor([W, H, W, H])).numpy()     # full-image pixels

# ---------------- 5) Visualize: original | SAM 3 input | result | result on original ----------------
img_disp = np.asarray(image)[::DISPLAY_DIV, ::DISPLAY_DIV][:h, :w]
z_disp   = z[::DISPLAY_DIV, ::DISPLAY_DIV][:h, :w]               # raw heights, unscaled
N = len(masks)
colors = plt.get_cmap("tab20")(np.linspace(0, 1, max(N, 1)))[:, :3].astype(np.float32)

# Blend all masks into one overlay layer
prem = np.zeros((h, w, 3), np.float32)
acc  = np.zeros((h, w), np.float32)
for i in range(N):
    m = masks[i]
    if m.any():
        prem[m] = ALPHA * colors[i] + (1 - ALPHA) * prem[m]
        acc[m]  = ALPHA + (1 - ALPHA) * acc[m]
overlay = np.zeros((h, w, 4), np.float32)
cov = acc > 0
overlay[cov, :3] = prem[cov] / acc[cov, None]
overlay[..., 3] = acc

# Object score = combined score / presence score  (e.g. 0.005 / 0.006 = 0.8)
if presence:                                   # not None and not 0
    object_scores = kept_scores / presence
else:
    object_scores = kept_scores                # no presence score: combined = object score

def draw_boxes(ax, label_scores, fmt=".2f"):
    """Draw boxes, score labels and the optional example box on one panel."""
    s = 1 / DISPLAY_DIV
    for i, (x1, y1, x2, y2) in enumerate(boxes_px):
        ax.add_patch(patches.Rectangle((x1 * s, y1 * s), (x2 - x1) * s, (y2 - y1) * s,
                                       fill=False, edgecolor=colors[i], linewidth=1.2))
        ax.text(x1 * s, y1 * s - 3, f"{float(label_scores[i]):{fmt}}", color=colors[i], fontsize=7)
    if example_box is not None:
        x1, y1, x2, y2 = example_box
        ax.add_patch(patches.Rectangle((x1 * s, y1 * s), (x2 - x1) * s, (y2 - y1) * s,
                                       fill=False, edgecolor="white", linestyle="--", linewidth=1.2))

# Figure size follows the image's width/height ratio
PANEL_H = 4.0                                      # inches per panel
GAP_IN  = 0.8                                      # space between panels, inches
panel_w = PANEL_H * w / h
fig, axes = plt.subplots(1, 4, figsize=(4 * panel_w + 3 * GAP_IN + 0.6, PANEL_H + 0.9),
                         layout="constrained")
fig.get_layout_engine().set(wspace=GAP_IN / panel_w)

raw_kw = dict(cmap="gray", vmin=0, vmax=float(z_disp.max()), aspect="equal")

# 1) Original raw height map
axes[0].imshow(z_disp, **raw_kw)
axes[0].set_title("Original\n(raw height map)", fontsize=8)

# 2) Input to SAM 3
axes[1].imshow(img_disp, aspect="equal")
axes[1].set_title("Input to SAM 3\n(scaled)", fontsize=8)

# 3) Result on the SAM 3 input, combined score (object x presence)
axes[2].imshow(img_disp, aspect="equal")
axes[2].imshow(overlay, aspect="equal")
draw_boxes(axes[2], kept_scores, fmt=".3f")
axes[2].set_title(f'"{text_prompt}": {N} instance(s)' + (" [forced]" if forced else "")
                  + "\ncombined score", fontsize=8)

# 4) Result on the original raw height map, object score (combined / presence)
axes[3].imshow(z_disp, **raw_kw)
axes[3].imshow(overlay, aspect="equal")
draw_boxes(axes[3], object_scores, fmt=".2f")
axes[3].set_title("Result on original image\n"
                  + (f"object score (presence {presence:.3f})" if presence else "object score"),
                  fontsize=8)

for ax in axes:
    ax.axis("off")
plt.show()

In [ ]:
import torch, numpy as np, torch.nn.functional as Fnn
import matplotlib.pyplot as plt, matplotlib.patches as patches
from PIL import Image

# ---------------- Settings ----------------
image_path  = "images/zil_00-10-83-13-99_261012_112259_Z.png"
text_prompt = "electronic component"    # "usb connector", "button"
example_box = None                      # e.g. [x1, y1, x2, y2] in full-image pixels -> positive example
THR         = 0.46                      # score threshold
REL_THR     = None                      # e.g. 0.5 -> keep scores >= 0.5 * best score (overrides THR)
MIN_ONE     = True                      # if nothing passes, keep the single best candidate
DISPLAY_DIV = 4                         # 1 = full resolution masks; 4 = 1/4 size (faster, less RAM)
ALPHA       = 0.6

# ---------------- 1) Load height map and scale to 0..255 ----------------
raw = Image.open(image_path)
print("file mode:", raw.mode)           # I;16 = 16-bit heights, L = already 8-bit
z = np.array(raw).astype(np.float32)
if z.ndim == 3:                         # already RGB: use one channel
    z = z[..., 0]
valid = z > 0                           # 0 = no data (laser shadow)
lo, hi = np.percentile(z[valid], [1, 99.5])
g = np.clip((z - lo) / (hi - lo), 0, 1)
g[~valid] = 0
image = Image.fromarray((g * 255).astype(np.uint8)).convert("RGB")
W, H = image.size

# ---------------- 2) Run SAM 3 ----------------
proc_kwargs = dict(images=image, text=text_prompt, return_tensors="pt")
if example_box is not None:
    proc_kwargs.update(input_boxes=[[example_box]], input_boxes_labels=[[1]])
inputs = processor(**proc_kwargs).to(model.device)

with torch.inference_mode():
    outputs = model(**inputs)

# Scores = object score x presence score (same formula as the processor)
scores = outputs.pred_logits[0].float().sigmoid()
presence = None
if outputs.presence_logits is not None:
    presence = float(outputs.presence_logits[0].float().sigmoid())
    scores = scores * presence
scores = scores.cpu()

order = scores.argsort(descending=True)
print("presence score:", None if presence is None else round(presence, 3))
print("top 10 scores:", np.round(scores[order[:10]].numpy(), 3))

# ---------------- 3) Choose which instances to keep ----------------
if REL_THR is not None:
    keep = torch.nonzero(scores >= REL_THR * scores.max()).squeeze(1)
else:
    keep = torch.nonzero(scores > THR).squeeze(1)
forced = False
if keep.numel() == 0 and MIN_ONE:
    keep = order[:1]                    # fall back to the best candidate
    forced = True
keep = keep[scores[keep].argsort(descending=True)]   # highest score first
print(f"kept {len(keep)} instance(s)" + ("  (forced top-1, check it!)" if forced else ""))

# Copy what we need to the CPU, then free the GPU
mask_logits = outputs.pred_masks[0, keep].float().cpu()          # (K, h, w), low resolution
boxes_norm  = outputs.pred_boxes[0, keep].float().cpu()          # x1, y1, x2, y2, normalized 0..1
kept_scores = scores[keep]
best_box_px = (outputs.pred_boxes[0, order[0]].float().cpu() * torch.tensor([W, H, W, H])).tolist()
print("best box (px):", [round(v) for v in best_box_px])

del outputs, inputs
torch.cuda.empty_cache()

# ---------------- 4) Scale masks up on the CPU, one at a time ----------------
h, w = H // DISPLAY_DIV, W // DISPLAY_DIV
masks = np.zeros((len(keep), h, w), dtype=bool)
for i in range(len(keep)):
    m = Fnn.interpolate(mask_logits[i][None, None], size=(h, w), mode="bilinear", align_corners=False)
    masks[i] = (m[0, 0] > 0).numpy()                             # logit > 0 == probability > 0.5
masks &= valid[::DISPLAY_DIV, ::DISPLAY_DIV][:h, :w]             # drop no-data pixels from masks
boxes_px = (boxes_norm * torch.tensor([W, H, W, H])).numpy()     # full-image pixels

# ---------------- 5) Visualize: original | SAM 3 input | result | result on original ----------------
img_disp = np.asarray(image)[::DISPLAY_DIV, ::DISPLAY_DIV][:h, :w]
z_disp   = z[::DISPLAY_DIV, ::DISPLAY_DIV][:h, :w]               # raw heights, unscaled
N = len(masks)
colors = plt.get_cmap("tab20")(np.linspace(0, 1, max(N, 1)))[:, :3].astype(np.float32)

# Blend all masks into one overlay layer
prem = np.zeros((h, w, 3), np.float32)
acc  = np.zeros((h, w), np.float32)
for i in range(N):
    m = masks[i]
    if m.any():
        prem[m] = ALPHA * colors[i] + (1 - ALPHA) * prem[m]
        acc[m]  = ALPHA + (1 - ALPHA) * acc[m]
overlay = np.zeros((h, w, 4), np.float32)
cov = acc > 0
overlay[cov, :3] = prem[cov] / acc[cov, None]
overlay[..., 3] = acc

# Object score = combined score / presence score  (e.g. 0.005 / 0.006 = 0.8)
if presence:                                   # not None and not 0
    object_scores = kept_scores / presence
else:
    object_scores = kept_scores                # no presence score: combined = object score

def draw_boxes(ax, label_scores, fmt=".2f"):
    """Draw boxes, score labels and the optional example box on one panel."""
    s = 1 / DISPLAY_DIV
    for i, (x1, y1, x2, y2) in enumerate(boxes_px):
        ax.add_patch(patches.Rectangle((x1 * s, y1 * s), (x2 - x1) * s, (y2 - y1) * s,
                                       fill=False, edgecolor=colors[i], linewidth=1.2))
        ax.text(x1 * s, y1 * s - 3, f"{float(label_scores[i]):{fmt}}", color=colors[i], fontsize=7)
    if example_box is not None:
        x1, y1, x2, y2 = example_box
        ax.add_patch(patches.Rectangle((x1 * s, y1 * s), (x2 - x1) * s, (y2 - y1) * s,
                                       fill=False, edgecolor="white", linestyle="--", linewidth=1.2))

# Figure size follows the image's width/height ratio
PANEL_H = 4.0                                      # inches per panel
GAP_IN  = 0.8                                      # space between panels, inches
panel_w = PANEL_H * w / h
fig, axes = plt.subplots(1, 4, figsize=(4 * panel_w + 3 * GAP_IN + 0.6, PANEL_H + 0.9),
                         layout="constrained")
fig.get_layout_engine().set(wspace=GAP_IN / panel_w)

raw_kw = dict(cmap="gray", vmin=0, vmax=float(z_disp.max()), aspect="equal")

# 1) Original raw height map
axes[0].imshow(z_disp, **raw_kw)
axes[0].set_title("Original\n(raw height map)", fontsize=8)

# 2) Input to SAM 3
axes[1].imshow(img_disp, aspect="equal")
axes[1].set_title("Input to SAM 3\n(scaled)", fontsize=8)

# 3) Result on the SAM 3 input, combined score (object x presence)
axes[2].imshow(img_disp, aspect="equal")
axes[2].imshow(overlay, aspect="equal")
draw_boxes(axes[2], kept_scores, fmt=".3f")
axes[2].set_title(f'"{text_prompt}": {N} instance(s)' + (" [forced]" if forced else "")
                  + "\ncombined score", fontsize=8)

# 4) Result on the original raw height map, object score (combined / presence)
axes[3].imshow(z_disp, **raw_kw)
axes[3].imshow(overlay, aspect="equal")
draw_boxes(axes[3], object_scores, fmt=".2f")
axes[3].set_title("Result on original image\n"
                  + (f"object score (presence {presence:.3f})" if presence else "object score"),
                  fontsize=8)

for ax in axes:
    ax.axis("off")
plt.show()

In [ ]:
import torch, numpy as np, torch.nn.functional as Fnn
import matplotlib.pyplot as plt, matplotlib.patches as patches
from PIL import Image

# ---------------- Settings ----------------
image_path  = "images/zil_00-01-14-66-99_261112_141735_Z_roll.png"
text_prompt = "scanned object"       
example_box = None                      # e.g. [x1, y1, x2, y2] in full-image pixels -> positive example
THR         = 0.46                      # score threshold
REL_THR     = None                      # e.g. 0.5 -> keep scores >= 0.5 * best score (overrides THR)
MIN_ONE     = True                      # if nothing passes, keep the single best candidate
DISPLAY_DIV = 4                         # 1 = full resolution masks; 4 = 1/4 size (faster, less RAM)
ALPHA       = 0.6

# ---------------- 1) Load height map and scale to 0..255 ----------------
raw = Image.open(image_path)
print("file mode:", raw.mode)           # I;16 = 16-bit heights, L = already 8-bit
z = np.array(raw).astype(np.float32)
if z.ndim == 3:                         # already RGB: use one channel
    z = z[..., 0]
valid = z > 0                           # 0 = no data (laser shadow)
lo, hi = np.percentile(z[valid], [1, 99.5])
g = np.clip((z - lo) / (hi - lo), 0, 1)
g[~valid] = 0
image = Image.fromarray((g * 255).astype(np.uint8)).convert("RGB")
W, H = image.size

# ---------------- 2) Run SAM 3 ----------------
proc_kwargs = dict(images=image, text=text_prompt, return_tensors="pt")
if example_box is not None:
    proc_kwargs.update(input_boxes=[[example_box]], input_boxes_labels=[[1]])
inputs = processor(**proc_kwargs).to(model.device)

with torch.inference_mode():
    outputs = model(**inputs)

# Scores = object score x presence score (same formula as the processor)
scores = outputs.pred_logits[0].float().sigmoid()
presence = None
if outputs.presence_logits is not None:
    presence = float(outputs.presence_logits[0].float().sigmoid())
    scores = scores * presence
scores = scores.cpu()

order = scores.argsort(descending=True)
print("presence score:", None if presence is None else round(presence, 3))
print("top 10 scores:", np.round(scores[order[:10]].numpy(), 3))

# ---------------- 3) Choose which instances to keep ----------------
if REL_THR is not None:
    keep = torch.nonzero(scores >= REL_THR * scores.max()).squeeze(1)
else:
    keep = torch.nonzero(scores > THR).squeeze(1)
forced = False
if keep.numel() == 0 and MIN_ONE:
    keep = order[:1]                    # fall back to the best candidate
    forced = True
keep = keep[scores[keep].argsort(descending=True)]   # highest score first
print(f"kept {len(keep)} instance(s)" + ("  (forced top-1, check it!)" if forced else ""))

# Copy what we need to the CPU, then free the GPU
mask_logits = outputs.pred_masks[0, keep].float().cpu()          # (K, h, w), low resolution
boxes_norm  = outputs.pred_boxes[0, keep].float().cpu()          # x1, y1, x2, y2, normalized 0..1
kept_scores = scores[keep]
best_box_px = (outputs.pred_boxes[0, order[0]].float().cpu() * torch.tensor([W, H, W, H])).tolist()
print("best box (px):", [round(v) for v in best_box_px])

del outputs, inputs
torch.cuda.empty_cache()

# ---------------- 4) Scale masks up on the CPU, one at a time ----------------
h, w = H // DISPLAY_DIV, W // DISPLAY_DIV
masks = np.zeros((len(keep), h, w), dtype=bool)
for i in range(len(keep)):
    m = Fnn.interpolate(mask_logits[i][None, None], size=(h, w), mode="bilinear", align_corners=False)
    masks[i] = (m[0, 0] > 0).numpy()                             # logit > 0 == probability > 0.5
masks &= valid[::DISPLAY_DIV, ::DISPLAY_DIV][:h, :w]             # drop no-data pixels from masks
boxes_px = (boxes_norm * torch.tensor([W, H, W, H])).numpy()     # full-image pixels

# ---------------- 5) Visualize: original | SAM 3 input | result | result on original ----------------
img_disp = np.asarray(image)[::DISPLAY_DIV, ::DISPLAY_DIV][:h, :w]
z_disp   = z[::DISPLAY_DIV, ::DISPLAY_DIV][:h, :w]               # raw heights, unscaled
N = len(masks)
colors = plt.get_cmap("tab20")(np.linspace(0, 1, max(N, 1)))[:, :3].astype(np.float32)

# Blend all masks into one overlay layer
prem = np.zeros((h, w, 3), np.float32)
acc  = np.zeros((h, w), np.float32)
for i in range(N):
    m = masks[i]
    if m.any():
        prem[m] = ALPHA * colors[i] + (1 - ALPHA) * prem[m]
        acc[m]  = ALPHA + (1 - ALPHA) * acc[m]
overlay = np.zeros((h, w, 4), np.float32)
cov = acc > 0
overlay[cov, :3] = prem[cov] / acc[cov, None]
overlay[..., 3] = acc

# Object score = combined score / presence score  (e.g. 0.005 / 0.006 = 0.8)
if presence:                                   # not None and not 0
    object_scores = kept_scores / presence
else:
    object_scores = kept_scores                # no presence score: combined = object score

def draw_boxes(ax, label_scores, fmt=".2f"):
    """Draw boxes, score labels and the optional example box on one panel."""
    s = 1 / DISPLAY_DIV
    for i, (x1, y1, x2, y2) in enumerate(boxes_px):
        ax.add_patch(patches.Rectangle((x1 * s, y1 * s), (x2 - x1) * s, (y2 - y1) * s,
                                       fill=False, edgecolor=colors[i], linewidth=1.2))
        ax.text(x1 * s, y1 * s - 3, f"{float(label_scores[i]):{fmt}}", color=colors[i], fontsize=7)
    if example_box is not None:
        x1, y1, x2, y2 = example_box
        ax.add_patch(patches.Rectangle((x1 * s, y1 * s), (x2 - x1) * s, (y2 - y1) * s,
                                       fill=False, edgecolor="white", linestyle="--", linewidth=1.2))

# Figure size follows the image's width/height ratio
PANEL_H = 4.0                                      # inches per panel
GAP_IN  = 0.8                                      # space between panels, inches
panel_w = PANEL_H * w / h
fig, axes = plt.subplots(1, 4, figsize=(4 * panel_w + 3 * GAP_IN + 0.6, PANEL_H + 0.9),
                         layout="constrained")
fig.get_layout_engine().set(wspace=GAP_IN / panel_w)

raw_kw = dict(cmap="gray", vmin=0, vmax=float(z_disp.max()), aspect="equal")

# 1) Original raw height map
axes[0].imshow(z_disp, **raw_kw)
axes[0].set_title("Original\n(raw height map)", fontsize=8)

# 2) Input to SAM 3
axes[1].imshow(img_disp, aspect="equal")
axes[1].set_title("Input to SAM 3\n(scaled)", fontsize=8)

# 3) Result on the SAM 3 input, combined score (object x presence)
axes[2].imshow(img_disp, aspect="equal")
axes[2].imshow(overlay, aspect="equal")
draw_boxes(axes[2], kept_scores, fmt=".3f")
axes[2].set_title(f'"{text_prompt}": {N} instance(s)' + (" [forced]" if forced else "")
                  + "\ncombined score", fontsize=8)

# 4) Result on the original raw height map, object score (combined / presence)
axes[3].imshow(z_disp, **raw_kw)
axes[3].imshow(overlay, aspect="equal")
draw_boxes(axes[3], object_scores, fmt=".2f")
axes[3].set_title("Result on original image\n"
                  + (f"object score (presence {presence:.3f})" if presence else "object score"),
                  fontsize=8)

for ax in axes:
    ax.axis("off")
plt.show()